### Calculate t cell emissions 
- Binary: is contacting another T cell
- Binary: is contacting another cancer cell
- Continuous: Instantaneous velocity
- If overlap morphologies look good
    - Area
    - Circularity

- Optional
    - Binary: is contacting a cancer cell that divides at some point in its life

- Backup features
    - Backup: Cancer cell division within 20um radius
    - Number of T cell neighbors within 20um radius
    - Number of cancer cell neighbors within 20um radius

In [1]:
import os
os.chdir("/gladstone/engelhardt/lab/adamw/MarsonImagingPipeline")

import pickle
import argparse
from pathlib import Path
import zipfile
from io import BytesIO

import yaml
import tifffile
import numpy as np
from scripts.utils.StatUtils import compute_cell_counts_per_frame, compute_mean_velocity_per_frame, calculate_area, compute_mean_cell_stat_per_frame, count_divisions_per_frame, compute_cell_velocities_per_frame_dict, compute_cell_cell_contact_dict, compute_all_cell_cell_contact_dict
from scripts.utils.CellTypingUtils import filter_tracks
from scripts.utils.StatUtils import *
from scripts.utils.PlottingUtils import *

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np

from scipy import stats

### Load tracks

In [2]:
# using Justin's directory for the config now, will be able to switch to my directory after the PipelineRestructure PR goes through
config_path = Path("/gladstone/engelhardt/lab/jadjasu/LiveCellUmbrella/MarsonImagingPipeline/snakemake_configs/experiment.yml")
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

cvat_base_dir = "/gladstone/engelhardt/lab/adamw/MarsonImagingPipeline/data/ground_truth_tracking_annotations/cvat_annotations/TCR-T/"
caliban_base_dir = '/gladstone/engelhardt/lab/MarsonLabIncucyteData/groundTruthCalibanTracks/'

crop_ids = ['B4_t50t100y200y350x750x900',
            'B8_t50t100y200y350x750x900',
            'E4_t50t100y200y350x750x900',
            'B4_t250t300y200y350x750x900',
            'B8_t250t300y200y350x750x900',
            'E4_t250t300y200y350x750x900']

conditions_dict = {}

conditions_dict["SH"] = ['B4_t50t100y200y350x750x900', 'B4_t250t300y200y350x750x900']
conditions_dict["RASA2"] = ['E4_t50t100y200y350x750x900', 'E4_t250t300y200y350x750x900']
conditions_dict["CUL5"] = ['B8_t50t100y200y350x750x900', 'B8_t250t300y200y350x750x900']

In [3]:
def get_gt_info_per_well(conditions):

    # if not config.get("CONDITIONS"):
    #     raise ValueError("No conditions found in config file.")
    
    # conditions = conditions_dict

    cvat_tracks_per_well = {}
    nuclear_tracks_per_well = {}
    cell_type_dict_per_well = {}

    for condition in conditions:
        for crop in conditions[condition]:
            well_id = crop.split('_')[0]
            cvat_tracks_per_well[crop] = tifffile.imread(os.path.join(cvat_base_dir, well_id, crop, 'ALL_tracks.tiff'))

            # get nuclear tracks
            caliban_tracks = tifffile.imread(os.path.join(caliban_base_dir, f'{crop}.tiff'))
            caliban_tracks = caliban_tracks[...,0]
            nuclear_tracks_per_well[crop] = caliban_tracks
            
            # get cell type dict
            cell_type_dict_per_well[crop] = pickle.load(open(os.path.join(cvat_base_dir, well_id, crop, f'full_cell_type_dict.pkl'), "rb"))
    
    return cvat_tracks_per_well, nuclear_tracks_per_well, cell_type_dict_per_well


In [4]:
# get relevant information
cvat_tracks_per_well, nuclear_tracks_per_well, cell_type_dict_per_well = get_gt_info_per_well(conditions_dict)

### Calculate relevant statistics

In [5]:
# get tracks by type for each well
type_tracks_per_well = {}
for cell_type in ['cancer', 't_cell', 'nuclei']:
    type_tracks_per_well[cell_type] = {}

    if cell_type == 'nuclei':
        type_tracks_per_well[cell_type] = nuclear_tracks_per_well.copy()
    else:
        type_tracks_per_well[cell_type] = {well: filter_tracks(cell_type, cvat_tracks_per_well[well], cell_type_dict_per_well[well]) for well in cvat_tracks_per_well.keys()}

In [9]:
t_cell_tracks = type_tracks_per_well['t_cell']

In [10]:
# calculate t cell velocities
t_cell_velocities_per_frame = {well: compute_cell_velocities_per_frame_dict(t_cell_tracks[well], unit_per_frame=config.get("TIME_FACTOR", 1)) for well in t_cell_tracks.keys()}


Computing cell velocities:   0%|          | 0/49 [00:00<?, ?it/s]

Computing cell velocities: 100%|██████████| 49/49 [00:00<00:00, 287.41it/s]


In [11]:
# calculate type-specific interactions

cancer_type_specific_contacts_per_frame, t_cell_type_specific_contacts_per_frame = {}, {}
cancer_type_specific_neighbors_per_frame, t_cell_type_specific_neighbors_per_frame = {}, {}

for well in cvat_tracks_per_well.keys():
    t_cell_tracks = type_tracks_per_well["t_cell"][well]
    cancer_tracks = type_tracks_per_well["cancer"][well]

    well_cancer_contacts_per_frame, well_t_cell_contacts_per_frame = compute_cell_cell_contact_dict(t_cell_tracks, cancer_tracks)
    well_cancer_neighbors_per_frame, well_t_cell_neighbors_per_frame = compute_cell_cell_neighbor_dict(t_cell_tracks, cancer_tracks)

    cancer_type_specific_contacts_per_frame[well] = well_cancer_contacts_per_frame
    t_cell_type_specific_contacts_per_frame[well] = well_t_cell_contacts_per_frame

    cancer_type_specific_neighbors_per_frame[well] = well_cancer_neighbors_per_frame
    t_cell_type_specific_neighbors_per_frame[well] = well_t_cell_neighbors_per_frame

Computing cell-cell contact dataframe...


Processing frame-by-frame contact output: 100%|██████████| 50/50 [00:00<00:00, 107.77it/s]


Computing cell-cell neighbor dataframe...


Processing frame-by-frame neighbor output: 100%|██████████| 50/50 [00:00<00:00, 124.15it/s]


Computing cell-cell contact dataframe...


...

Processing frame-by-frame neighbor output: 100%|██████████| 50/50 [00:00<00:00, 170.04it/s]


Computing cell-cell contact dataframe...


Processing frame-by-frame contact output: 100%|██████████| 50/50 [00:00<00:00, 122.80it/s]


Computing cell-cell neighbor dataframe...


Processing frame-by-frame neighbor output: 100%|██████████| 50/50 [00:00<00:00, 147.10it/s]


In [12]:
# calculate type-agnostic interactions

cancer_all_contacts_per_frame, t_cell_all_contacts_per_frame = {}, {}
cancer_all_neighbors_per_frame, t_cell_all_neighbors_per_frame = {}, {}

for well in cvat_tracks_per_well.keys():
    t_cell_tracks = type_tracks_per_well["t_cell"][well]
    cancer_tracks = type_tracks_per_well["cancer"][well]

    well_cancer_contacts_per_frame, well_t_cell_contacts_per_frame = compute_all_cell_cell_contact_dict(t_cell_tracks, cancer_tracks)
    well_cancer_neighbors_per_frame, well_t_cell_neighbors_per_frame = compute_all_cell_cell_neighbor_dict(t_cell_tracks, cancer_tracks)

    cancer_all_contacts_per_frame[well] = well_cancer_contacts_per_frame
    t_cell_all_contacts_per_frame[well] = well_t_cell_contacts_per_frame

    cancer_all_neighbors_per_frame[well] = well_cancer_neighbors_per_frame
    t_cell_all_neighbors_per_frame[well] = well_t_cell_neighbors_per_frame

Computing cell-cell contact dataframe...


Processing contact dict: 100%|██████████| 50/50 [00:00<00:00, 97.19it/s] 


Computing cell-cell neighbor dataframe...


Processing neighbor dict: 100%|██████████| 50/50 [00:00<00:00, 105.22it/s]


Computing cell-cell contact dataframe...


...

Processing neighbor dict: 100%|██████████| 50/50 [00:00<00:00, 113.00it/s]


Computing cell-cell contact dataframe...


Processing contact dict: 100%|██████████| 50/50 [00:00<00:00, 74.26it/s]


Computing cell-cell neighbor dataframe...


Processing neighbor dict: 100%|██████████| 50/50 [00:00<00:00, 78.52it/s]


### Create a single emissions array per crop

In [13]:
crop_ids

['B4_t50t100y200y350x750x900',
 'B8_t50t100y200y350x750x900',
 'E4_t50t100y200y350x750x900',
 'B4_t250t300y200y350x750x900',
 'B8_t250t300y200y350x750x900',
 'E4_t250t300y200y350x750x900']

In [15]:
# loop over crop IDs
for crop in crop_ids:

    # subset to the desired tracks 
    example_cvat_tracks = cvat_tracks_per_well[crop]
    example_t_cell_tracks = type_tracks_per_well["t_cell"][crop]
    example_cell_type_dict = cell_type_dict_per_well[crop]

    example_t_cell_velocities_per_frame = t_cell_velocities_per_frame[crop]
    example_t_cell_type_specific_neighbors_per_frame = t_cell_type_specific_neighbors_per_frame[crop]
    example_t_cell_all_neighbors_per_frame = t_cell_all_neighbors_per_frame[crop]

    t_cell_ids = np.unique(example_t_cell_tracks[example_t_cell_tracks > 0])
    id_to_column_index = {cell_id: index for index, cell_id in enumerate(t_cell_ids)}

    emissions_array = np.zeros((50, len(t_cell_ids), 3)) # shape: (num_frames, num_t_cells, num_emission_features)

    for frame, velocities in example_t_cell_velocities_per_frame.items():
        for cell_id, velocity in velocities.items():
            if cell_id in id_to_column_index:
                column_index = id_to_column_index[cell_id]
                emissions_array[frame, column_index, 0] = velocity
    
    for frame in example_t_cell_all_neighbors_per_frame.keys():
        all_neighbors = example_t_cell_all_neighbors_per_frame[frame]
        type_specific_neighbors = example_t_cell_type_specific_neighbors_per_frame[frame]

        for cell_id, neighbors in all_neighbors.items():
            #print(f"frame: {frame}, cell_id: {cell_id}, neighbors: {neighbors}")
            if cell_id in id_to_column_index:
                column_index = id_to_column_index[cell_id]

                cancer_neighbors_list = type_specific_neighbors.get(cell_id, [0])
                cancer_neighbors = cancer_neighbors_list[0]

                t_cell_neighbors = neighbors - cancer_neighbors

                emissions_array[frame, column_index, 1] = cancer_neighbors
                emissions_array[frame, column_index, 2] = t_cell_neighbors

    # save the emissions computed for this crop
    out_dir = f'/gladstone/engelhardt/lab/adamw/treeHMM/notebooks/data/example_gt_crop/{crop}'
    os.makedirs(out_dir, exist_ok=True)
    np.save(os.path.join(out_dir, 't_cell_emissions_array.npy'), emissions_array)

In [16]:
emissions_array.shape

(50, 105, 3)